# Machine Learning Pipeline for Customer Churn Prediction

This notebook demonstrates the end-to-end ML pipeline using TensorFlow Extended (TFX) and Apache Beam for the Dicoding MLOps final project.

## 1. Imports and Setup

In [ ]:
import os
from tfx import v1 as tfx
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext

PIPELINE_NAME = 'muhammad_adha-pipeline'
PIPELINE_ROOT = os.path.join(os.getcwd(), PIPELINE_NAME)
METADATA_PATH = os.path.join(PIPELINE_ROOT, 'metadata.sqlite')
SERVING_MODEL_DIR = os.path.join(os.getcwd(), 'app', 'model_store')
DATA_ROOT = os.path.join(os.getcwd(), 'data')

In [ ]:
context = InteractiveContext(pipeline_root=PIPELINE_ROOT)

## 2. ExampleGen

In [ ]:
output = tfx.proto.Output(
    split_config=tfx.proto.SplitConfig(splits=[
        tfx.proto.SplitConfig.Split(name='train', hash_buckets=8),
        tfx.proto.SplitConfig.Split(name='eval', hash_buckets=2)
    ])
)
example_gen = tfx.components.CsvExampleGen(input_base=DATA_ROOT, output_config=output)
context.run(example_gen)

## 3. StatisticsGen

In [ ]:
statistics_gen = tfx.components.StatisticsGen(
    examples=example_gen.outputs['examples']
)
context.run(statistics_gen)

## 4. SchemaGen

In [ ]:
schema_gen = tfx.components.SchemaGen(
    statistics=statistics_gen.outputs['statistics'],
    infer_feature_shape=True
)
context.run(schema_gen)

## 5. ExampleValidator

In [ ]:
example_validator = tfx.components.ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)
context.run(example_validator)

## 6. Transform

In [ ]:
TRANSFORM_MODULE_FILE = 'modules/transform_module.py'

transform = tfx.components.Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file=TRANSFORM_MODULE_FILE
)
context.run(transform)

## 7. Tuner

In [ ]:
TUNER_MODULE_FILE = 'modules/tuner_module.py'

tuner = tfx.components.Tuner(
    module_file=TUNER_MODULE_FILE,
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=tfx.proto.TrainArgs(splits=['train'], num_steps=20),
    eval_args=tfx.proto.EvalArgs(splits=['eval'], num_steps=5)
)
context.run(tuner)

## 8. Trainer

In [ ]:
TRAINER_MODULE_FILE = 'modules/trainer_module.py'

trainer = tfx.components.Trainer(
    module_file=TRAINER_MODULE_FILE,
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    hyperparameters=tuner.outputs['best_hyperparameters'],
    train_args=tfx.proto.TrainArgs(splits=['train'], num_steps=100),
    eval_args=tfx.proto.EvalArgs(splits=['eval'], num_steps=20)
)
context.run(trainer)

## 9. Resolver

In [ ]:
model_resolver = tfx.dsl.Resolver(
    strategy_class=tfx.dsl.experimental.LatestBlessedModelStrategy,
    model=trainer.outputs['model'],
    model_blessing=tfx.dsl.Channel(type=tfx.types.standard_artifacts.ModelBlessing)
).with_id('latest_blessed_model_resolver')
context.run(model_resolver)

## 10. Evaluator

In [ ]:
import tensorflow_model_analysis as tfma

eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key='churn')],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(metrics=[
            tfma.MetricConfig(class_name='ExampleCount'),
            tfma.MetricConfig(class_name='AUC'),
            tfma.MetricConfig(class_name='FalsePositives'),
            tfma.MetricConfig(class_name='TruePositives'),
            tfma.MetricConfig(class_name='FalseNegatives'),
            tfma.MetricConfig(class_name='TrueNegatives'),
            tfma.MetricConfig(class_name='BinaryAccuracy',
                threshold=tfma.MetricThreshold(
                    value_threshold=tfma.GenericValueThreshold(
                        lower_bound={'value': 0.5}),
                    change_threshold=tfma.GenericChangeThreshold(
                        direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                        absolute={'value': -1e-10})))
        ])
    ]
)

evaluator = tfx.components.Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config
)
context.run(evaluator)

## 11. Pusher

In [ ]:
pusher = tfx.components.Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=tfx.proto.PushDestination(
        filesystem=tfx.proto.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    )
)
context.run(pusher)